<a href="https://colab.research.google.com/github/cagg991010/Introduccion-Ing-Datos/blob/main/notebooks/TP2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import os

def ejecutar_script_completo():
    # Nombre de la base de datos SQLite
    db_name = "tp2_base_datos.db"

    # Si existe una base previa, la eliminamos para empezar de cero
    if os.path.exists(db_name):
        os.remove(db_name)
        print(f"Base de datos anterior '{db_name}' eliminada para una prueba limpia.")

    # Conexión a la base de datos
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # Habilitar soporte para claves foráneas en SQLite
    cursor.execute("PRAGMA foreign_keys = ON;")

    print("\n--- 1. CREANDO ESQUEMA RELACIONAL OLTP (PUNTO 1) ---")

    sql_oltp = """
    -- Catálogo Geográfico
    CREATE TABLE Ubicacion (
        id_ubicacion INTEGER PRIMARY KEY AUTOINCREMENT,
        codigo_pais VARCHAR(10) NOT NULL,
        nombre_pais VARCHAR(100) NOT NULL,
        nombre_lugar VARCHAR(150),
        latitud DECIMAL(9,6),
        longitud DECIMAL(9,6)
    );

    -- Catálogo de Medios de Comunicación
    CREATE TABLE Fuente_Medio (
        id_fuente INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre_fuente VARCHAR(150) NOT NULL UNIQUE,
        tipo_fuente VARCHAR(50)
    );

    -- Catálogo de Actores e Instituciones
    CREATE TABLE Actor (
        id_actor INTEGER PRIMARY KEY AUTOINCREMENT,
        codigo_actor VARCHAR(20) NOT NULL UNIQUE,
        nombre_actor VARCHAR(100),
        tipo_actor VARCHAR(50),
        codigo_pais_actor VARCHAR(10)
    );

    -- Catálogo Taxonómico CAMEO
    CREATE TABLE Codigo_CAMEO (
        codigo_cameo VARCHAR(10) PRIMARY KEY,
        descripcion_cameo VARCHAR(200) NOT NULL,
        categoria_raiz VARCHAR(100) NOT NULL
    );

    -- Tabla Central de Eventos Periodísticos
    CREATE TABLE Noticia_Evento (
        id_evento INTEGER PRIMARY KEY AUTOINCREMENT,
        global_event_id BIGINT UNIQUE NOT NULL,
        fecha_evento DATE NOT NULL,
        id_ubicacion INTEGER NOT NULL,
        id_fuente INTEGER NOT NULL,
        num_mentions INTEGER DEFAULT 1 CHECK (num_mentions >= 0),
        num_sources INTEGER DEFAULT 1 CHECK (num_sources >= 0),
        num_articles INTEGER DEFAULT 1 CHECK (num_articles >= 0),
        avg_tone FLOAT DEFAULT 0.0,
        url_noticia VARCHAR(500),
        FOREIGN KEY (id_ubicacion) REFERENCES Ubicacion(id_ubicacion) ON DELETE RESTRICT,
        FOREIGN KEY (id_fuente) REFERENCES Fuente_Medio(id_fuente) ON DELETE RESTRICT
    );

    -- Tabla Asociativa Noticia <-> Actor (M:N)
    CREATE TABLE Evento_Actor (
        id_evento INTEGER NOT NULL,
        id_actor INTEGER NOT NULL,
        rol_actor VARCHAR(20) NOT NULL,
        PRIMARY KEY (id_evento, id_actor, rol_actor),
        FOREIGN KEY (id_evento) REFERENCES Noticia_Evento(id_evento) ON DELETE CASCADE,
        FOREIGN KEY (id_actor) REFERENCES Actor(id_actor) ON DELETE RESTRICT
    );

    -- Tabla Asociativa Noticia <-> Categoría CAMEO (M:N - Cobertura Multi-Temática)
    CREATE TABLE Evento_CAMEO (
        id_evento INTEGER NOT NULL,
        codigo_cameo VARCHAR(10) NOT NULL,
        es_codigo_principal BOOLEAN NOT NULL DEFAULT 1,
        PRIMARY KEY (id_evento, codigo_cameo),
        FOREIGN KEY (id_evento) REFERENCES Noticia_Evento(id_evento) ON DELETE CASCADE,
        FOREIGN KEY (codigo_cameo) REFERENCES Codigo_CAMEO(codigo_cameo) ON DELETE RESTRICT
    );
    """
    cursor.executescript(sql_oltp)
    print("✓ Tablas OLTP creadas exitosamente.")

    print("\n--- 2. CREANDO ESQUEMA DIMENSIONAL OLAP (PUNTO 2) ---")

    sql_olap = """
    -- Dimensión Tiempo
    CREATE TABLE Dim_Tiempo (
        id_tiempo INTEGER PRIMARY KEY,
        fecha DATE NOT NULL,
        anio INTEGER NOT NULL,
        trimestre INTEGER NOT NULL CHECK (trimestre BETWEEN 1 AND 4),
        mes INTEGER NOT NULL CHECK (mes BETWEEN 1 AND 12),
        nombre_mes VARCHAR(20) NOT NULL,
        dia_semana VARCHAR(20) NOT NULL,
        es_fin_de_semana BOOLEAN NOT NULL,
        es_festivo BOOLEAN NOT NULL
    );

    -- Dimensión Ubicación
    CREATE TABLE Dim_Ubicacion (
        id_ubicacion INTEGER PRIMARY KEY,
        codigo_pais VARCHAR(10) NOT NULL,
        nombre_pais VARCHAR(100) NOT NULL,
        nombre_lugar VARCHAR(150),
        latitud DECIMAL(9,6),
        longitud DECIMAL(9,6)
    );

    -- Dimensión Fuente / Medio
    CREATE TABLE Dim_Fuente (
        id_fuente INTEGER PRIMARY KEY,
        nombre_fuente VARCHAR(150) NOT NULL,
        tipo_fuente VARCHAR(50)
    );

    -- Dimensión Actor
    CREATE TABLE Dim_Actor (
        id_actor INTEGER PRIMARY KEY,
        codigo_actor VARCHAR(20) NOT NULL,
        nombre_actor VARCHAR(100),
        tipo_actor VARCHAR(50),
        codigo_pais_actor VARCHAR(10)
    );

    -- Dimensión Categoría CAMEO
    CREATE TABLE Dim_CAMEO (
        codigo_cameo VARCHAR(10) PRIMARY KEY,
        descripcion_cameo VARCHAR(200) NOT NULL,
        categoria_raiz VARCHAR(100) NOT NULL
    );

    -- Tabla Puente para Dimensión Multi-Valuada CAMEO
    CREATE TABLE Bridge_Grupo_CAMEO (
        id_grupo_cameo INTEGER NOT NULL,
        codigo_cameo VARCHAR(10) NOT NULL,
        factor_ponderacion DECIMAL(3,2) NOT NULL CHECK (factor_ponderacion > 0 AND factor_ponderacion <= 1),
        PRIMARY KEY (id_grupo_cameo, codigo_cameo),
        FOREIGN KEY (codigo_cameo) REFERENCES Dim_CAMEO(codigo_cameo)
    );

    -- Tabla de Hechos Central
    CREATE TABLE Fact_Cobertura_Medios (
        id_hecho INTEGER PRIMARY KEY AUTOINCREMENT,
        id_evento BIGINT NOT NULL, -- Dimensión Degenerada
        id_tiempo INTEGER NOT NULL,
        id_ubicacion INTEGER NOT NULL,
        id_fuente INTEGER NOT NULL,
        id_actor INTEGER NOT NULL,
        id_grupo_cameo INTEGER NOT NULL,
        num_mentions INTEGER NOT NULL CHECK (num_mentions >= 0),
        num_sources INTEGER NOT NULL CHECK (num_sources >= 0),
        num_articles INTEGER NOT NULL CHECK (num_articles >= 0),
        avg_tone FLOAT NOT NULL,
        cantidad_eventos INTEGER DEFAULT 1,
        FOREIGN KEY (id_tiempo) REFERENCES Dim_Tiempo(id_tiempo),
        FOREIGN KEY (id_ubicacion) REFERENCES Dim_Ubicacion(id_ubicacion),
        FOREIGN KEY (id_fuente) REFERENCES Dim_Fuente(id_fuente),
        FOREIGN KEY (id_actor) REFERENCES Dim_Actor(id_actor)
    );
    """
    cursor.executescript(sql_olap)
    print("✓ Tablas OLAP (Esquema Estrella + Tabla Puente) creadas exitosamente.")

    print("\n--- 3. INSERTANDO DATOS DE PRUEBA (DML) ---")

    sql_datos = """
    -- Datos de prueba para OLTP
    INSERT INTO Ubicacion (codigo_pais, nombre_pais, nombre_lugar, latitud, longitud) VALUES
    ('ARG', 'Argentina', 'Buenos Aires', -34.6037, -58.3816),
    ('USA', 'Estados Unidos', 'Washington D.C.', 38.9072, -77.0369);

    INSERT INTO Fuente_Medio (nombre_fuente, tipo_fuente) VALUES
    ('nytimes.com', 'Prensa Digital'),
    ('clarin.com', 'Periódico Local');

    INSERT INTO Actor (codigo_actor, nombre_actor, tipo_actor, codigo_pais_actor) VALUES
    ('USA', 'United States', 'Gobierno', 'USA'),
    ('ARGGOV', 'Gobierno de Argentina', 'Gobierno', 'ARG'),
    ('NGO', 'Human Rights Watch', 'ONG', 'GLO');

    INSERT INTO Codigo_CAMEO (codigo_cameo, descripcion_cameo, categoria_raiz) VALUES
    ('010', 'Make public statement', 'Verbal Cooperation'),
    ('020', 'Appeal for material cooperation', 'Verbal Cooperation'),
    ('190', 'Use unconventional violence', 'Material Conflict');

    INSERT INTO Noticia_Evento (global_event_id, fecha_evento, id_ubicacion, id_fuente, num_mentions, num_sources, num_articles, avg_tone, url_noticia) VALUES
    (10000001, '2026-09-15', 1, 2, 15, 3, 5, 2.50, 'https://clarin.com/nota1'),
    (10000002, '2026-09-20', 2, 1, 40, 8, 12, -4.20, 'https://nytimes.com/article2');

    INSERT INTO Evento_Actor (id_evento, id_actor, rol_actor) VALUES
    (1, 1, 'Actor1'), (1, 2, 'Actor2'),
    (2, 1, 'Actor1'), (2, 3, 'Actor2');

    INSERT INTO Evento_CAMEO (id_evento, codigo_cameo, es_codigo_principal) VALUES
    (1, '010', 1), (1, '020', 0),
    (2, '190', 1);

    -- Datos de prueba para OLAP
    INSERT INTO Dim_Tiempo VALUES
    (20260915, '2026-09-15', 2026, 3, 9, 'Septiembre', 'Martes', 0, 0),
    (20260920, '2026-09-20', 2026, 3, 9, 'Septiembre', 'Domingo', 1, 0);

    INSERT INTO Dim_Ubicacion VALUES (1, 'ARG', 'Argentina', 'Buenos Aires', -34.6037, -58.3816);
    INSERT INTO Dim_Ubicacion VALUES (2, 'USA', 'Estados Unidos', 'Washington D.C.', 38.9072, -77.0369);

    INSERT INTO Dim_Fuente VALUES (1, 'nytimes.com', 'Prensa Digital');
    INSERT INTO Dim_Fuente VALUES (2, 'clarin.com', 'Periódico Local');

    INSERT INTO Dim_Actor VALUES (1, 'USA', 'United States', 'Gobierno', 'USA');
    INSERT INTO Dim_Actor VALUES (2, 'ARGGOV', 'Gobierno de Argentina', 'Gobierno', 'ARG');
    INSERT INTO Dim_Actor VALUES (3, 'NGO', 'Human Rights Watch', 'ONG', 'GLO');

    INSERT INTO Dim_CAMEO VALUES ('010', 'Make public statement', 'Verbal Cooperation');
    INSERT INTO Dim_CAMEO VALUES ('020', 'Appeal for material cooperation', 'Verbal Cooperation');
    INSERT INTO Dim_CAMEO VALUES ('190', 'Use unconventional violence', 'Material Conflict');

    -- Tabla Puente: Noticia 1 tiene 2 temas CAMEO -> factor_ponderacion = 0.50 cada uno
    INSERT INTO Bridge_Grupo_CAMEO VALUES (1, '010', 0.50);
    INSERT INTO Bridge_Grupo_CAMEO VALUES (1, '020', 0.50);
    INSERT INTO Bridge_Grupo_CAMEO VALUES (2, '190', 1.00);

    INSERT INTO Fact_Cobertura_Medios (id_evento, id_tiempo, id_ubicacion, id_fuente, id_actor, id_grupo_cameo, num_mentions, num_sources, num_articles, avg_tone) VALUES
    (10000001, 20260915, 1, 2, 1, 1, 15, 3, 5, 2.50),
    (10000002, 20260920, 2, 1, 3, 2, 40, 8, 12, -4.20);
    """
    cursor.executescript(sql_datos)
    conn.commit()
    print("✓ Datos cargados correctamente.")

    print("\n--- 4. EJECUTANDO CONSULTAS ANALÍTICAS DE NEGOCIO ---")

    # Consulta 1: Rendimiento e Impacto
    q1 = """
    SELECT u.nombre_pais, t.nombre_mes, SUM(f.num_mentions) AS total_menciones, SUM(f.num_articles) AS total_articulos
    FROM Fact_Cobertura_Medios f
    JOIN Dim_Ubicacion u ON f.id_ubicacion = u.id_ubicacion
    JOIN Dim_Tiempo t ON f.id_tiempo = t.id_tiempo
    GROUP BY u.nombre_pais, t.nombre_mes;
    """
    print("\n📊 Pregunta 1: Menciones y Artículos por País y Mes:")
    for row in cursor.execute(q1).fetchall():
        print(f"   País: {row[0]} | Mes: {row[1]} | Menciones: {row[2]} | Artículos: {row[3]}")

    # Consulta 2: Sentimiento por Tipo de Actor
    q2 = """
    SELECT a.tipo_actor, ROUND(AVG(f.avg_tone), 2) AS tono_promedio
    FROM Fact_Cobertura_Medios f
    JOIN Dim_Actor a ON f.id_actor = a.id_actor
    GROUP BY a.tipo_actor;
    """
    print("\n📊 Pregunta 2: Sentimiento Promedio (Tono) por Tipo de Actor:")
    for row in cursor.execute(q2).fetchall():
        print(f"   Tipo de Actor: {row[0]} | Tono Promedio: {row[1]}")

    # Consulta 3: Días Hábiles vs Fin de Semana
    q3 = """
    SELECT
        CASE WHEN t.es_fin_de_semana = 1 THEN 'Fin de Semana' ELSE 'Día Hábil' END AS tipo_dia,
        SUM(f.num_articles) AS total_articulos
    FROM Fact_Cobertura_Medios f
    JOIN Dim_Tiempo t ON f.id_tiempo = t.id_tiempo
    GROUP BY t.es_fin_de_semana;
    """
    print("\n📊 Pregunta 3: Cobertura en Días Hábiles vs. Fines de Semana:")
    for row in cursor.execute(q3).fetchall():
        print(f"   Tipo de Día: {row[0]} | Artículos Publicados: {row[1]}")

    # Consulta 4: Cobertura Multi-Temática con Prorrateo mediante Tabla Puente
    q4 = """
    SELECT c.descripcion_cameo, SUM(f.num_articles * b.factor_ponderacion) AS articulos_prorrateados
    FROM Fact_Cobertura_Medios f
    JOIN Bridge_Grupo_CAMEO b ON f.id_grupo_cameo = b.id_grupo_cameo
    JOIN Dim_CAMEO c ON b.codigo_cameo = c.codigo_cameo
    GROUP BY c.descripcion_cameo;
    """
    print("\n📊 Pregunta 4: Artículos Prorrateados por Categoría CAMEO (Tabla Puente):")
    for row in cursor.execute(q4).fetchall():
        print(f"   Tema CAMEO: {row[0]} | Artículos Prorrateados: {row[1]}")

    conn.close()
    print("\n✅ Proceso completado exitosamente. La base de datos local 'tp2_base_datos.db' se encuentra lista.")

In [2]:
if __name__ == "__main__":
    ejecutar_script_completo()


--- 1. CREANDO ESQUEMA RELACIONAL OLTP (PUNTO 1) ---
✓ Tablas OLTP creadas exitosamente.

--- 2. CREANDO ESQUEMA DIMENSIONAL OLAP (PUNTO 2) ---
✓ Tablas OLAP (Esquema Estrella + Tabla Puente) creadas exitosamente.

--- 3. INSERTANDO DATOS DE PRUEBA (DML) ---
✓ Datos cargados correctamente.

--- 4. EJECUTANDO CONSULTAS ANALÍTICAS DE NEGOCIO ---

📊 Pregunta 1: Menciones y Artículos por País y Mes:
   País: Argentina | Mes: Septiembre | Menciones: 15 | Artículos: 5
   País: Estados Unidos | Mes: Septiembre | Menciones: 40 | Artículos: 12

📊 Pregunta 2: Sentimiento Promedio (Tono) por Tipo de Actor:
   Tipo de Actor: Gobierno | Tono Promedio: 2.5
   Tipo de Actor: ONG | Tono Promedio: -4.2

📊 Pregunta 3: Cobertura en Días Hábiles vs. Fines de Semana:
   Tipo de Día: Día Hábil | Artículos Publicados: 5
   Tipo de Día: Fin de Semana | Artículos Publicados: 12

📊 Pregunta 4: Artículos Prorrateados por Categoría CAMEO (Tabla Puente):
   Tema CAMEO: Appeal for material cooperation | Artículos 